<a href="https://colab.research.google.com/github/duaf9877/FlyRank-AI-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duaf9877/FlyRank-AI-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page should be reviewed for refresh if it has not been updated for a long time and still receives meaningful search visibility. Older pages with good visibility are more likely to benefit from a content refresh than pages with little traffic.

Reason Codes
stale_but_visible – Old content with strong visibility.
stale_low_position – Old content that has slipped in rankings.
high_visibility – High impressions but not yet stale.
low_priority – Does not meet any important condition.

# Signal Check 1 – Staleness

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

stale_table = (
    df.assign(
        stale_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[-1,90,180,365,100000],
            labels=["0-90","91-180","181-365","365+"]
        )
    )
    .groupby("stale_bucket")
    .agg(
        n=("content_id","count"),
        decline_rate=("trend_direction", lambda x: (x=="down").mean())
    )
)

print(stale_table)

                  n  decline_rate
stale_bucket                     
0-90          20655      0.512031
91-180         9171      0.611057
181-365         169      0.467456
365+              5      0.600000


/tmp/ipykernel_693/2340882225.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("stale_bucket")


# Signal Check 2 – Search Visibility

In [ ]:
visibility_table = (
    df.assign(
        impression_bucket=pd.cut(
            df["impressions_90d"],
            bins=[-1,100,500,2000,np.inf],
            labels=["0-100","101-500","501-2000","2000+"]
        )
    )
    .groupby("impression_bucket")
    .agg(
        n=("content_id","count"),
        decline_rate=("trend_direction", lambda x: (x=="down").mean())
    )
)

print(visibility_table)

                       n  decline_rate
impression_bucket                     
0-100               8006      0.389208
101-500             5279      0.604281
501-2000            6502      0.617964
2000+              10213      0.581416


/tmp/ipykernel_693/1262290806.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("impression_bucket")


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("content_refresh_anonymized.csv")

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["low_position"] = (
    (df["avg_position"] >= 10) &
    (df["avg_position"] != 0)
).astype(int)

df["score"] = (
    2 * df["stale"] +
    1 * df["visible"] +
    1 * df["low_position"]
)

def reason_code(row):
    if row["stale"] and row["visible"]:
        return "stale_but_visible"
    elif row["stale"] and row["low_position"]:
        return "stale_low_position"
    elif row["visible"]:
        return "high_visibility"
    else:
        return "low_priority"

df["reason_code"] = df.apply(reason_code, axis=1)

df["action"] = np.where(
    df["score"] >= 3,
    "refresh_content",
    "monitor"
)

queue = (
    df.sort_values(
        ["score","impressions_90d"],
        ascending=[False,False]
    )
)

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(queue.head(20)[[
    "content_id",
    "score",
    "action",
    "reason_code",
    "impressions_90d",
    "days_since_last_update",
    "avg_position"
]])

                 content_id  score           action         reason_code  \
16751  content_cf56e2e2e282      4  refresh_content   stale_but_visible   
16514  content_7368877ea310      4  refresh_content   stale_but_visible   
7021   content_1bfaa38ff26c      4  refresh_content   stale_but_visible   
21268  content_0a91db491d14      4  refresh_content   stale_but_visible   
11489  content_5feee3994adb      4  refresh_content   stale_but_visible   
12045  content_c2d929d83eaa      4  refresh_content   stale_but_visible   
698    content_b16bd7307b39      4  refresh_content   stale_but_visible   
5327   content_fe16a55cd13d      4  refresh_content   stale_but_visible   
26810  content_ecb6215e79fd      4  refresh_content   stale_but_visible   
20837  content_928af3e22c80      4  refresh_content   stale_but_visible   
23215  content_bdbec75c1148      4  refresh_content   stale_but_visible   
26799  content_77d4d5930e5e      4  refresh_content   stale_but_visible   
11630  content_6226ee6adc

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue.head(20)

for rank, (_, row) in enumerate(top20.iterrows(), start=1):

    print(f"{rank}.")
    print(f"Action: {row['action']}")
    print(f"Reason: {row['reason_code']}")

    if row["score"] >= 3:
        confidence = "High"
    elif row["score"] == 2:
        confidence = "Medium"
    else:
        confidence = "Low"

    print(f"Confidence: {confidence}")

    print(
        "What would make it wrong: "
        "Recent content changes, seasonal traffic variation, "
        "or missing business context that is not captured by the dataset."
    )

    print("-"*60)

1.
Action: refresh_content
Reason: stale_but_visible
Confidence: High
What would make it wrong: Recent content changes, seasonal traffic variation, or missing business context that is not captured by the dataset.
------------------------------------------------------------
2.
Action: refresh_content
Reason: stale_but_visible
Confidence: High
What would make it wrong: Recent content changes, seasonal traffic variation, or missing business context that is not captured by the dataset.
------------------------------------------------------------
3.
Action: refresh_content
Reason: stale_but_visible
Confidence: High
What would make it wrong: Recent content changes, seasonal traffic variation, or missing business context that is not captured by the dataset.
------------------------------------------------------------
4.
Action: refresh_content
Reason: stale_but_visible
Confidence: High
What would make it wrong: Recent content changes, seasonal traffic variation, or missing business context th

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks

Some highly ranked pages may be weak recommendations because:

The page may have seasonal traffic patterns.
High impressions do not always mean a content refresh is needed.
Ranking changes could be caused by external events rather than stale content.
Some pages may already have been refreshed recently outside the reporting window.
Leakage check

No product flags or future-window information were used.

The baseline only used:

days_since_last_update
impressions_90d
avg_position

The following columns were not used because they are label-derived or would cause leakage:

trend_direction
trend_pct

These columns were only used to evaluate the exploratory signal checks and were not used when computing the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.